# 08 — Streamlit Dashboard

Launches an interactive Streamlit web-app on Colab via pyngrok.

**Three tabs:**
| Tab | What it does |
|-----|--------------|
| 🔬 Predictions | Pick D / pitch/D / Q → find nearest HDF5 case → run MultiHeadMGN → show T, TMA, p, |U| field slices + guardrail check |
| 📊 Optimizer | Scatter-plot Pareto front from `optimizer_results.json`; top-10 candidate table |
| ⚠️ Guardrails | Batch confidence histogram + violation table from `guardrail_results.json` |

**Requires:**
- Colab A100 GPU runtime
- `07_optimizer.ipynb` completed (writes `optimizer_results.json`)
- Free ngrok account — get token at ngrok.com and paste into Cell 3

## 0. Setup

In [ ]:
import subprocess, sys, os

subprocess.run(['pip', 'install', '-q',
                'torch_geometric', 'h5py', 'scipy',
                'streamlit', 'pyngrok', 'plotly'], check=True)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/Ranaam21/cfd-ald-app.git'
REPO_DIR = '/content/cfd-ald-app'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(r.stdout.strip() or 'Repo up to date.')
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned.')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Setup done.')

## 1. Verify paths

In [ ]:
from pathlib import Path

DRIVE_BASE  = Path('/content/drive/MyDrive/cfd-ald-app')
HDF5_DIR    = DRIVE_BASE / 'showerhead_openfoam'
MULTIHEAD   = DRIVE_BASE / 'checkpoints' / 'multihead' / 'multihead_final.pt'
OPT_JSON    = DRIVE_BASE / 'checkpoints' / 'optimizer' / 'optimizer_results.json'
GR_JSON     = DRIVE_BASE / 'checkpoints' / 'guardrail' / 'guardrail_results.json'

for label, p in [('HDF5 dir',       HDF5_DIR),
                 ('Multihead ckpt', MULTIHEAD),
                 ('Optimizer JSON', OPT_JSON),
                 ('Guardrail JSON', GR_JSON)]:
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  [{status}]  {label}: {p}')

h5_files = sorted(HDF5_DIR.glob('case_*.h5'))
print(f'\n  Found {len(h5_files)} HDF5 cases')

## 2. Verify app.py

`app.py` lives in the repo root — git pull in Cell 0 already brought it in. Just confirm it's there.

In [ ]:
import os
app_path = '/content/cfd-ald-app/app.py'
assert os.path.exists(app_path), f'Missing: {app_path} — did Cell 0 (git pull) run?'
print(f'app.py found ({os.path.getsize(app_path)} bytes) — ready to launch.')

## 3. Launch Streamlit + ngrok

Get a **free** ngrok auth token at https://ngrok.com → Dashboard → Auth → Copy token.  
Paste it into `NGROK_TOKEN` below.

In [ ]:
import subprocess, time, threading

# Kill any previous streamlit/tunnel processes
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
subprocess.run(['pkill', '-f', 'lt --port'],  capture_output=True)
time.sleep(2)

# Start Streamlit
st_proc = subprocess.Popen(
    ['streamlit', 'run', '/content/cfd-ald-app/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(5)
if st_proc.poll() is not None:
    print('ERROR — Streamlit failed:')
    print(st_proc.stderr.read().decode())
else:
    print('Streamlit running on port 8501.')

# Expose via localtunnel (no account needed)
subprocess.run(['npm', 'install', '-g', 'localtunnel'], capture_output=True)
lt_proc = subprocess.Popen(
    ['npx', 'localtunnel', '--port', '8501'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
)
time.sleep(3)
url_line = lt_proc.stdout.readline().strip()
print(f'\nStreamlit URL: {url_line}')
print('Open the URL → if prompted for a password, enter your Colab VM IP:')
import urllib.request
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
print(f'  Tunnel password (if asked): {ip}')

## 4. Stop server (run when done)

In [ ]:
from pyngrok import ngrok as _ng
_ng.disconnect_all()
st_proc.terminate()
print('Server stopped.')